In [1]:
import pathlib

from saver.technicals.twse import stocks

root = pathlib.Path(r'C:\Users\Atlas\Desktop\Storage\02-Projects\stocklab\playground\data\init\technicals\TWSE\stocks\1st')
stocks.add_source(root)
stocks.add_database_path('','stock')
# df = stocks.execute_single('20040211.json')[1]

ModuleNotFoundError: No module named 'saver'

In [ ]:
import database
import pathlib

root = pathlib.Path(r'C:\Users\Atlas\Desktop\Storage\02-Projects\stocklab\playground\data\init\technicals\TWSE\stocks\1st')

commander = database.commander.Commander('test.db')

item = database.saver.items.technicals.twse.stocks
item.add_source(root)

# commander.init_table(item.table)
# commander.update_table(item.table)

item.save_bacth(commander)


['ALTER TABLE stocks ADD COLUMN others TEXT', 'CREATE INDEX IF NOT EXISTS idx_date ON stocks (date);']


['ALTER TABLE stocks ADD COLUMN others TEXT',
 'CREATE INDEX IF NOT EXISTS idx_date ON stocks (date);']

In [ ]:
from bs4 import BeautifulSoup
import requests
import re

stock_id = '2308'

url = f"https://ic.tpex.org.tw/company_chain.php?stk_code={stock_id}"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers, timeout=10)
response.encoding = 'utf-8'

soup = BeautifulSoup(response.text, 'html.parser')
table = soup.find('body').find('center').find('div', 'main-panel').find('div', 'content-panel-main').find('div', 'content').find_all('h4')


[l.text.replace('►','').replace('\xa0','').split('>') for l in table[1:]]



[['休閒娛樂', '休閒車業'],
 ['半導體', '生產製程及檢測設備'],
 ['半導體', 'IC模組'],
 ['半導體', 'IC通路'],
 ['電腦及週邊設備', '電源供應器'],
 ['電腦及週邊設備', '散熱片、風扇馬達、散熱模組'],
 ['電腦及週邊設備', '筆記型電腦'],
 ['電腦及週邊設備', '桌上型電腦'],
 ['電腦及週邊設備', '精簡型電腦'],
 ['電腦及週邊設備', '安全監控系統'],
 ['平面顯示器', '其他零組件'],
 ['通信網路', '網路設備(如數據機、網路卡、閘道器、路由器、網路電話)'],
 ['通信網路', '光通訊設備(如光纖電纜、光傳輸設備)'],
 ['通信網路', '無線通訊設備(如行動電話、衛星定位系統、衛星通訊設備、微波通訊設備、數位機上盒)'],
 ['被動元件', '電阻器'],
 ['被動元件', '電容器'],
 ['被動元件', '電感器'],
 ['電機機械', '傳動元件'],
 ['電機機械', '電控元件'],
 ['電機機械', '專用機械(如紡織成衣生產用機械、食品飲料生產用機械、農林用機械等)'],
 ['電機機械', '輸送機械及零配件'],
 ['軟體服務', '應用/系統軟體設計開發'],
 ['建材營造', '機電工程'],
 ['其他', '特殊照明業'],
 ['汽車', '其他'],
 ['人工智慧', '系統整合'],
 ['人工智慧', '顧問諮詢'],
 ['人工智慧', '領域解決方案'],
 ['人工智慧', '智慧設備'],
 ['人工智慧', '機器學習'],
 ['人工智慧', '電腦視覺'],
 ['人工智慧', '運算設備'],
 ['雲端運算', '電力設備'],
 ['雲端運算', '冷卻設備'],
 ['雲端運算', '設備管理軟體'],
 ['雲端運算', '系統整合'],
 ['雲端運算', '顧問諮詢'],
 ['雲端運算', '設備安裝服務'],
 ['大數據', '系統整合'],
 ['大數據', '顧問諮詢'],
 ['大數據', '領域解決方案'],
 ['大數據', '運算元件與設備'],
 ['半導體', 'LED驅動IC'],
 ['半導體', '消費性IC'],
 ['半導體', '微控制

In [61]:
from database.sources.technicals.otc import stocks_stage_1
import pathlib

root = pathlib.Path( r'C:\Users\Atlas\Desktop\Storage\02-Projects\stocklab\playground\data\init\technicals\OTC\stocks\stage1\1st')

name = '20060228'
# name = '20030818'
# name = '20040202'
# name = '20041028'

path = root.joinpath(name).with_suffix('.html')
content = stocks_stage_1.open(path)

from bs4 import BeautifulSoup
import pandas as pd
import re

content, file = content

if file.stem < '20041028':
    content = BeautifulSoup(content, "html.parser")
    tables = content.find('body').find_all('table')

    dfs = []

    head_cols = ['代號','證券名稱','收盤價','漲跌','開盤價','最高價','最低價','均價','成交股數','成交金額','成交筆數','最後委買價','最後委賣價']
    head_idx = 4
    for table in tables:
        lines = re.sub(r'\n+', '\n', table.text.replace('<RSTA3104>','')).replace(' ','').split('＊＊＊＊＊管理股票＊＊＊＊＊')[0].strip().split('\n')
        lines = [l.strip() for l in lines]
        columns = lines[head_idx:head_idx+len(head_cols)]

        data = lines[head_idx+len(head_cols):-1]
        data = [data[i:i+len(head_cols)] for i in range(0,len(data),len(head_cols))]

        if file.stem < '20040202':
            assert columns == head_cols
        else:
            for drop, d in enumerate(data):
                if not re.search(r'\d', d[0]):
                    break
            data = data[:drop]
    
        df = pd.DataFrame(columns=head_cols, data=data)
        
        if file.stem >= '20040202':
            df['證券名稱'] = 'lost'

        dfs.append(df)
    complete_df = pd.concat(dfs, axis=0)
else:
    tables = pd.read_html(file, encoding='utf-8')
    assert len(tables) == 1
    table = tables[0]
    table.columns = table.loc[0]
    table = table.loc[1:]

    if file.stem < '20041125':
        head_cols = [
            '股票 代號', '證券 名稱', '收盤價', '漲跌', '漲跌', '開盤價', '最高價', '最低價', '均價', 
            '成交股數', '成交金額(元)', '成交筆數', '最後 委買價', '最後 委賣價', 
            '發行股數', '次日 參考價', '次日 漲停價', '次日 跌停價'
        ]
        assert table.columns.to_list() == head_cols
    else:
        head_cols = [
            '股票 代號', '證券 名稱', '收盤價', '收盤價', '漲跌', '漲跌', '開盤價', '最高價', '最低價', '均價', 
            '成交股數', '成交金額(元)', '成交筆數', '最後 委買價', '最後 委賣價', 
            '發行股數', '次日 參考價', '次日 漲停價', '次日 跌停價'
        ]
        assert table.columns.to_list() == head_cols
        head_cols[2] = '收盤價-drop'
        table.columns = head_cols

    mapping = {
        '股票 代號':'代號',
        '證券 名稱':'證券名稱',
        '收盤價':'收盤價',
        '開盤價':'開盤價',
        '最高價':'最高價',
        '最低價':'最低價',
        '成交股數':'成交股數',
        '成交金額(元)':'成交金額',
        '成交筆數':'成交筆數',
    }

    table = table[list(mapping.keys())]
    table.columns = [mapping[c] for c in table.columns]

    table = table.loc[(table['代號']!='管 理 股 票').cumprod() == 1]

    complete_df = table


# df = stocks_stage_1.standardize(content,name)